# Notebook 5: Rheological Models — Using dv/v and Geodetic Strain## Inverting for rheology from combined seismic and geodetic observations### Key References- Okubo, K. et al. (2024). JGR Solid Earth, 129, e2023JB028084.- Snieder, R. et al. (2017). Logarithmic healing model.- Clements, T. & Denolle, M. A. (2023). JGR Solid Earth, 128, e2022JB025553.- Sens-Schönfelder, C. & Eulenfeld, T. (2019). PRL, 122, 138501.### Physical FrameworkAfter earthquakes, velocity typically drops co-seismically and recovers logarithmically.The healing timescales and the relationship between dv/v and geodetic strain encode information about subsurface **rheology** — whether the response is purely elastic, viscoelastic, or involves slow dynamics (mesoscopic nonlinearity).

In [ ]:
import numpy as npimport matplotlib.pyplot as pltfrom scipy.integrate import quadplt.rcParams.update({    'font.size': 12, 'axes.labelsize': 14, 'figure.dpi': 150,    'font.family': 'serif', 'mathtext.fontset': 'cm',    'axes.grid': True, 'grid.alpha': 0.3})print("Environment ready.")

## 1. Logarithmic Healing Model (Snieder et al., 2017)Post-seismic velocity recovery follows (Okubo et al., 2024, Eq. 5):$$L(t, \tau_{\min}, \tau_{\max}, t_{EQ}) = -\int_{\tau_{\min}}^{\tau_{\max}} \frac{1}{\tau} \exp\left(-\frac{t - t_{EQ}}{\tau}\right) d\tau$$The coseismic velocity drop magnitude is: $s \cdot \ln(\tau_{\max}/\tau_{\min})$This model predicts:- **Initial rapid recovery** (governed by $\tau_{\min}$)- **Long-term slow recovery** (governed by $\tau_{\max}$)- The rate slows logarithmically: $\dot{v}/v \propto 1/t$

In [ ]:
# === Logarithmic Healing Model ===def healing_integral(t, tau_min, tau_max, t_eq):    """    Logarithmic healing model (Snieder et al., 2017).    Okubo et al. (2024), Eq. 5.    """    if t < t_eq:        return 0.0    dt = t - t_eq    if dt <= 0:        return 0.0        # Numerical integration    def integrand(tau):        return (1.0/tau) * np.exp(-dt/tau)        result, _ = quad(integrand, tau_min, tau_max)    return -resultdef healing_series(times, tau_min, tau_max, t_eq):    return np.array([healing_integral(t, tau_min, tau_max, t_eq) for t in times])# Time in dayst_days = np.linspace(0, 365*20, 5000)  # 20 yearst_eq = 365 * 2  # earthquake at year 2# Explore different healing timescalesfig, axes = plt.subplots(2, 2, figsize=(14, 10))# Panel (a): Effect of tau_maxax = axes[0, 0]tau_min_fixed = 0.1 * 365.25  # 0.1 years in daysfor tau_max_yr in [1, 10, 100, 1000, 10000]:    tau_max = tau_max_yr * 365.25    L = healing_series(t_days, tau_min_fixed, tau_max, t_eq)    coseismic = np.log(tau_max / tau_min_fixed)    L_normalized = L / coseismic  # normalize by coseismic drop    ax.plot(t_days/365.25, L_normalized, lw=2, label=f'$\\tau_{{max}}$ = {tau_max_yr} yr')ax.set_xlabel('Time [years]')ax.set_ylabel('Normalized healing')ax.set_title(f'(a) Effect of $\\tau_{{max}}$ ($\\tau_{{min}}$ = 0.1 yr)')ax.legend(fontsize=9)ax.axvline(t_eq/365.25, color='red', ls='--', alpha=0.5, label='Earthquake')# Panel (b): Effect of tau_minax = axes[0, 1]tau_max_fixed = 1000 * 365.25for tau_min_yr in [0.01, 0.1, 1.0, 5.0]:    tau_min = tau_min_yr * 365.25    L = healing_series(t_days, tau_min, tau_max_fixed, t_eq)    coseismic = np.log(tau_max_fixed / tau_min)    L_normalized = L / coseismic    ax.plot(t_days/365.25, L_normalized, lw=2, label=f'$\\tau_{{min}}$ = {tau_min_yr} yr')ax.set_xlabel('Time [years]')ax.set_ylabel('Normalized healing')ax.set_title(f'(b) Effect of $\\tau_{{min}}$ ($\\tau_{{max}}$ = 1000 yr)')ax.legend(fontsize=9)# Panel (c): Full Parkfield-like modelax = axes[1, 0]# San Simeon (2003) and Parkfield (2004) earthquakest_ss = 1.0 * 365.25  # year 1t_pf = 2.5 * 365.25  # year 2.5# Parameters inspired by Okubo et al. (2024)s1, tau_min1, tau_max1 = 0.2, 36.5, 10000*365.25s2, tau_max2 = 0.5, 5000*365.25tau_min2 = 36.5L1 = healing_series(t_days, tau_min1, tau_max1, t_ss)L2 = healing_series(t_days, tau_min2, tau_max2, t_pf)# Environmental terms (simplified)a0 = 0.0p2_temp = 0.05 * np.cos(2*np.pi*t_days/365.25)  # thermoelastic# Linear trendb0 = 0.005 / 365.25  # ~ 0.005 %/year (Okubo et al. 2024: 0.0048%/yr)dvv_model = a0 + s1*L1 + s2*L2 + p2_temp + b0*(t_days - t_days[0])ax.plot(t_days/365.25, dvv_model, 'k-', lw=1.5, label='Full model')ax.plot(t_days/365.25, s1*L1 + s2*L2, 'b--', lw=1, label='Healing only')ax.plot(t_days/365.25, p2_temp, 'g--', lw=1, alpha=0.5, label='Thermoelastic')ax.plot(t_days/365.25, b0*(t_days - t_days[0]), 'r--', lw=1, label='Linear trend (tectonic)')ax.axvline(t_ss/365.25, color='orange', ls=':', label='San Simeon')ax.axvline(t_pf/365.25, color='red', ls=':', label='Parkfield')ax.set_xlabel('Time [years]')ax.set_ylabel('dv/v [%] (synthetic)')ax.set_title('(c) Parkfield-like synthetic dv/v model')ax.legend(fontsize=8, ncol=2)# Panel (d): Healing rate in log-logax = axes[1, 1]dt = t_days - t_pfmask = dt > 1tau_min_log = 36.5tau_max_log = 5000*365.25L_log = healing_series(t_days[mask], tau_min_log, tau_max_log, t_pf)rate = np.gradient(L_log, dt[mask])ax.loglog(dt[mask]/365.25, np.abs(rate)*365.25, 'b-', lw=2)ax.set_xlabel('Time after earthquake [years]')ax.set_ylabel('Healing rate [1/year]')ax.set_title('(d) Healing rate (log-log → power law)')fig.suptitle('Logarithmic Healing & Rheological Models\n'             '(Snieder et al. 2017; Okubo et al. 2024)', fontsize=14, y=1.02)plt.tight_layout()plt.savefig('fig13_healing_models.png', bbox_inches='tight')plt.show()

## 2. Viscoelastic Rheology: Maxwell, Kelvin-Voigt, and SLSDifferent rheological models predict different relationships between dv/v and strain:| Model | Behavior | dv/v signature ||-------|----------|----------------|| **Elastic** | Instantaneous, reversible | $dv/v \propto \epsilon$ (linear) || **Maxwell** | Stress relaxation | Exponential decay of dv/v || **Kelvin-Voigt** | Strain retardation | Delayed dv/v response || **SLS** | Both relaxation + retardation | Frequency-dependent dv/v || **Slow dynamics** | Logarithmic recovery | $dv/v \propto \ln(t)$ after perturbation |

In [ ]:
# === Viscoelastic Rheology Comparison ===
def maxwell_response(t, tau, epsilon_0):
    """Maxwell model: stress relaxation under constant strain."""
    return epsilon_0 * np.exp(-t/tau)
def kelvin_voigt_response(t, tau, sigma_0):
    """Kelvin-Voigt: strain retardation under constant stress."""
    return sigma_0 * (1 - np.exp(-t/tau))
def sls_response(t, tau_sigma, tau_epsilon, epsilon_0):
    """Standard Linear Solid: combination."""
    M_R = 1.0  # normalized relaxed modulus
    M_U = M_R * tau_epsilon / tau_sigma  # unrelaxed modulus
    return epsilon_0 * (M_R + (M_U - M_R) * np.exp(-t/tau_sigma))
def slow_dynamics(t, A, tau_0=1):
    """Logarithmic slow dynamics."""
    return A * np.log(1 + t/tau_0)
t = np.linspace(0.01, 100, 1000)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
# Panel (a): Step response in time domain
ax = axes[0, 0]
ax.plot(t, maxwell_response(t, 10, 1), 'b-', lw=2, label='Maxwell ($\\tau$ = 10)')
ax.plot(t, maxwell_response(t, 30, 1), 'b--', lw=2, label='Maxwell ($\\tau$ = 30)')
ax.plot(t, kelvin_voigt_response(t, 10, 1), 'r-', lw=2, label='Kelvin-Voigt ($\\tau$ = 10)')
ax.plot(t, sls_response(t, 10, 30, 1), 'g-', lw=2, label='SLS ($\\tau_\\sigma$=10, $\\tau_\\epsilon$=30)')
ax.plot(t, slow_dynamics(t, 0.3), 'k--', lw=2, label='Slow dynamics (log)')
ax.set_xlabel('Time [normalized]')
ax.set_ylabel('Response [normalized]')
ax.set_title('(a) Rheological step responses')
ax.legend(fontsize=9)
# Panel (b): Frequency-dependent velocity (SLS)
ax = axes[1, 0]
omega = np.logspace(-3, 3, 500)
tau_values = [1, 5, 20, 100]
for tau in tau_values:
    # SLS complex modulus: M*(w) = M_R * (1 + i*w*tau_eps)/(1 + i*w*tau_sig)
    tau_sig = tau
    tau_eps = 3*tau
    M_star = (1 + 1j*omega*tau_eps) / (1 + 1j*omega*tau_sig)
    V_ratio = np.sqrt(np.abs(M_star))
    Q_inv = np.imag(M_star) / np.real(M_star)
    ax.semilogx(omega, V_ratio, lw=2, label=f'$\\tau_\\sigma$ = {tau}')
ax.set_xlabel('Angular frequency $\\omega$')
ax.set_ylabel('$V/V_0$')
ax.set_title('(b) SLS: frequency-dependent velocity')
ax.legend()
# Panel (c): dv/v vs strain crossplot for different rheologies
ax = axes[0, 1]
# Elastic: straight line
strain = np.linspace(-1, 1, 200)
beta = -500
ax.plot(strain, beta * strain * 1e-4, 'b-', lw=2.5, label='Elastic (instantaneous)')
# Viscoelastic: hysteresis loop
theta = np.linspace(0, 2*np.pi, 200)
strain_cycle = np.sin(theta)
dvv_viscoel = beta * strain_cycle * 1e-4 * np.cos(theta - 0.3)  # phase lag
ax.plot(strain_cycle, dvv_viscoel, 'r-', lw=2, label='Viscoelastic (phase lag)')
# Nonlinear: curved
dvv_nonlin = beta * strain * 1e-4 * (1 + 0.5 * strain)
ax.plot(strain, dvv_nonlin, 'g--', lw=2, label='Nonlinear elastic')
ax.set_xlabel('Strain [normalized]')
ax.set_ylabel('dv/v [normalized]')
ax.set_title('(c) dv/v–strain crossplots: rheology diagnostics')
ax.legend(fontsize=9)
ax.axhline(0, color='k', alpha=0.3)
ax.axvline(0, color='k', alpha=0.3)
# Panel (d): Tidal modulation of dv/v → probe in-situ nonlinearity
ax = axes[1, 1]
hours = np.linspace(0, 72, 1000)
# Earth tide strain
strain_tide = 50e-9 * np.sin(2*np.pi*hours/12.42) + 30e-9 * np.sin(2*np.pi*hours/25.82)
# Linear response
dvv_tide_lin = -500 * strain_tide
# Nonlinear response.
# Quadratic coefficient chosen so the nonlinear term is a visible (~20%) fraction
# of the linear term at the tidal strain amplitude (eps ~ 5e-8):
#   gamma*eps^2 ~ 0.2*|beta|*eps  =>  gamma ~ 0.2*|beta|/eps ~ 2e9.
# (A textbook gamma~1e5 would be ~1e4x too small to see at tidal strains.)
gamma_tide = 2e9
dvv_tide_nl = -500 * strain_tide + gamma_tide * strain_tide**2
ax.plot(hours, strain_tide*1e9, 'k-', lw=1, label='Tidal strain [nε]')
ax2 = ax.twinx()
ax2.plot(hours, dvv_tide_lin*1e6, 'b-', lw=1.5, alpha=0.7, label='Linear dv/v')
ax2.plot(hours, dvv_tide_nl*1e6, 'r-', lw=1.5, alpha=0.7, label='Nonlinear dv/v')
ax.set_xlabel('Time [hours]')
ax.set_ylabel('Strain [nanostrain]', color='k')
ax2.set_ylabel('dv/v [×10⁻⁶]', color='blue')
ax.set_title('(d) Tidal probing of nonlinearity\n(cf. Sens-Schönfelder & Eulenfeld, 2019)')
ax2.legend(fontsize=9, loc='upper right')
fig.suptitle('Rheological Models & Diagnostics for dv/v', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('fig14_rheology_comparison.png', bbox_inches='tight')
plt.show()
print("Key diagnostic: if dv/v-strain crossplot shows:")
print("  - Linear line → pure elastic (linear + nonlinear)")
print("  - Ellipse → viscoelastic (phase lag)")
print("  - Curvature → higher-order nonlinearity")
print("  - Hysteresis → slow dynamics / mesoscopic nonlinearity")